In [ ]:
from kaggle_environments import make

env = make("cabt")
env.run(["submission/main.py", "submission/main.py"])

with open("resulte.html", "w") as f:
    f.write(env.render(mode="html"))

print("Simulation finished.")
assert [s.status for s in env.state] == ["DONE", "DONE"], env.state
print("rewards:", [s.reward for s in env.state], "| decisions:", len(env.steps))

In [ ]:
from rl.eval import play_games

wr_self, _ = play_games("submission/main.py", "submission/main.py", 5,
                        replay_prefix="m0_selfplay", names=("m0-policy", "m0-policy"))
wr_rand, _ = play_games("submission/main.py", "random", 10,
                        replay_prefix="m0_vs_random", names=("m0-policy", "random"))
print(f"self-play: {wr_self:.0%}   vs random: {wr_rand:.0%}")

In [ ]:
import webbrowser, pathlib
webbrowser.open(pathlib.Path("replays/index.html").resolve().as_uri())

In [ ]:
from rl.gate import load_submission_module
from rl.eval import option_type_report, play_games

sub = load_submission_module()                 # the actual numpy submission agent
print(option_type_report(sub.agent, n_games=5))

wr, _ = play_games("submission/main.py", "random", 100, replay_prefix="m1_vs_random")
print(f"vs random: {wr:.0%}")

In [1]:
%load_ext autoreload
%autoreload 2
# Visit-concentration Sanity Check

import random, torch, numpy as np
from cg.api import to_observation_class
from cg.game import battle_start, battle_select, battle_finish

from rl.policy import OptionScorer
from rl.mcts import determinize, make_node, mcts_search

model = OptionScorer()
model.load_state_dict(torch.load("checkpoints/bc_v1.pt", map_location="cpu")); model.eval()
deck = [int(x) for x in open("decks/kyogre.csv") if x.strip()]

obs_dict, _ = battle_start(deck, deck)
for _ in range(14):
    if obs_dict["current"]["result"] >= 0: break
    o = obs_dict
    obs_dict = battle_select(random.sample(range(len(o["select"]["option"])), o["select"]["maxCount"]))

root = make_node(determinize(obs_dict if False else to_observation_class(obs_dict), deck), model)
visits = mcts_search(root, model, 32)

order = np.argsort(visits)[::-1]

print("prior : ", np.round(root.P, 3))
print("visits: ", visits.astype(int))
print("Q     : ", np.round(np.where(root.N>0, root.W/np.maximum(root.N,1), 0), 3))
print(f"argmax visits = action {order[0]}  |  argmax prior = action {int(np.argmax(root.P))}")
from cg.api import search_end; search_end(); battle_finish()


prior :  [0.999 0.001]
visits:  [27  5]
Q     :  [-0.857 -0.555]
argmax visits = action 0  |  argmax prior = action 0


In [3]:
from rl.eval import play_games
from rl.teacher import load_teacher
from rl.mcts import make_mcts_agent

MCTS = make_mcts_agent(model, deck, n_sims=100)
LUC = load_teacher("le", agent="lucario", deck="lucario")

wr, _ = play_games(MCTS, LUC, 30, replay_prefix="mcts_vs_luc", names=("MCTS(bc_v1)", "LUC-expert"))

print(f"MCTS(bc_v1) vs LUC-expert: {wr:.0%}   (plain bc_v1 got 22%)")

MCTS(bc_v1) vs LUC-expert: 27%   (plain bc_v1 got 22%)
